# Time Series Momentum, Backtested on Crypto
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikshev/unstuck/blob/main/episodes/q02-time-series-momentum/q02_time_series_momentum.ipynb)

Time-series momentum as published by Moskowitz, Ooi and Pedersen, *Time Series Momentum*,
Journal of Financial Economics (2012): every month, go long an asset whose past-12-month return
is positive and short one whose return is negative, each sized to the same volatility.
We test it on every coin that had a Binance perpetual in January 2020 — long and short,
with fees and funding. Education, not financial advice.

Two differences from the paper, stated up front: it uses *excess* returns (minus the risk-free rate),
we use plain returns; it annualises volatility with 261 trading days, crypto trades all 365.

## 1. Install
freqtrade is the backtester; we call only a few of its methods and explain each one.

In [ ]:
!pip install -q freqtrade==2026.10

## 2. Helpers from the earlier episodes
`binance_vision` — raw Binance data with SHA-256 hash checks (built in episode q00); `quant_report` — freqtrade's verdict and hourly marked-to-market equity (built in episode q01).

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py -O https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/quant_report.py
from binance_vision import save_for_freqtrade, DATA
from quant_report import verdict, equity, drawdown
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. The universe: every perpetual Binance listed by January 2020
Chosen by listing date, not by who is still around today — so it includes the EOS coin, which stopped trading in 2025. Uses `save_for_freqtrade`, `DATA` (section 2).

In [ ]:
# every coin with a Binance perpetual by January 2020 - survivors or not
coins = ["BTC", "ETH", "BCH", "XRP", "EOS", "LTC",
         "TRX", "ETC", "LINK", "XLM", "ADA"]
for c in coins:
    save_for_freqtrade(c + "USDT", "1d", "2020-01", "2026-10")
daily = str(DATA / "{}_USDT_USDT-1d-futures.feather")
close = pd.DataFrame({c: pd.read_feather(daily.format(c))
                         .set_index("date")["close"] for c in coins})
span = close.apply(lambda s: f"{s.first_valid_index().date()} to "
                             f"{s.last_valid_index().date()}")
print(span.to_string())

## 4. The signal: the sign of the past 12 months
Return over 365 days, $r_{12m} = P_t / P_{t-365} - 1$; long if it is positive, short if negative. Uses `close` (section 3).

In [ ]:
mom = close / close.shift(365) - 1                 # past-12-month return
# the last day of each month: the next day is the 1st
month_end = mom.index[(mom.index + pd.Timedelta(days=1)).day == 1]
signal = np.sign(mom.loc[month_end])               # +1 long, -1 short
print("BTC, last six month-ends:")
btc_view = pd.DataFrame({"12m return": mom.loc[month_end, "BTC"],
                         "signal": signal["BTC"]})
print(btc_view.tail(6).round(3).to_string())

## 5. The size: every coin gets the same risk
Ex-ante volatility $\sigma$ = exponentially weighted standard deviation of daily returns (centre of mass 60 days) × $\sqrt{365}$. Weight $= \dfrac{40\%}{\sigma} \cdot \dfrac{1}{S}$ of the account, S = the number of coins trading that day (the paper's $S_t$). Uses `close` (3), `month_end` (4).

In [ ]:
# paper: exponentially weighted, centre of mass 60 days; x sqrt(365) = a year
sigma = close.pct_change().ewm(com=60).std() * np.sqrt(365)
last = month_end[-1]
S = close.loc[last].notna().sum()                  # coins trading that day
weight = 0.40 / sigma / S                          # 40% target volatility
print("coins trading on", last.date(), ":", S)
table = pd.DataFrame({"volatility": sigma.loc[last],
                      "weight": weight.loc[last]})
print(table[close.loc[last].notna()].round(3).to_string())

## 6. Backtest settings
**Cross** margin: eleven small positions share one wallet, so one coin doubling does not liquidate a short on its own. Leverage 1, 10,000 USDT (Tether, a dollar stablecoin), the strategy sizes every position. Uses `coins` (section 3).

In [ ]:
config = {
    "trading_mode": "futures",
    "margin_mode": "cross",                  # one wallet backs every position
    "liquidation_buffer": 0.0,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",             # the strategy sizes each position
    "dry_run_wallet": 10000, "dry_run": True,
    "max_open_trades": 11,                   # one position per coin
    "timeframe": "1d", "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True,
                      "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True,
                     "order_book_top": 1},
    "exchange": {"name": "binance",
                 "pair_whitelist": [c + "/USDT:USDT" for c in coins]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 7. Strategy file, part 1: the settings
12-month lookback, 40% target volatility, leverage 1. freqtrade insists on a stoploss: −99% is as far as it goes.

In [ ]:
%%writefile user_data/strategies/TSMOM.py
import numpy as np
import pandas as pd
from freqtrade.strategy import IStrategy

class TSMOM(IStrategy):
    # paper: 12-month lookback, 40% target volatility; leverage 1; shorts on
    LOOKBACK, TARGET, LEV, SHORTS = 365, 0.40, 1, True
    timeframe = "1d"
    can_short = True
    minimal_roi = {"0": 100}            # no profit target: only the signal
    stoploss = -0.99                    # compulsory, as far as freqtrade allows
    position_adjustment_enable = True   # needed for the monthly resize
    startup_candle_count = 366          # a year of history before trade one

## 8. Part 2: indicators
The same volatility and 12-month return as sections 4–5, and a flag on the last day of each month. `LOOKBACK` from section 7.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def populate_indicators(self, df, metadata):
        returns = df["close"].pct_change()
        df["sigma"] = returns.ewm(com=60).std() * np.sqrt(365)   # section 5
        df["mom"] = df["close"] / df["close"].shift(self.LOOKBACK) - 1
        # decisions are made on the last close of each month
        df["month_end"] = (df["date"] + pd.Timedelta(days=1)).dt.day == 1
        return df

## 9. Part 3: entries and exits
On a month's last close: long if momentum is positive, short if negative; leave the position whose sign flipped. The order fills at the next open — the 1st. Columns from section 8, `SHORTS` from 7.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def populate_entry_trend(self, df, metadata):
        # the paper's rule: the sign of the past 12 months
        df.loc[df["month_end"] & (df["mom"] > 0), "enter_long"] = 1
        if self.SHORTS:
            df.loc[df["month_end"] & (df["mom"] < 0), "enter_short"] = 1
        return df

    def populate_exit_trend(self, df, metadata):
        # the sign flipped: leave, and the entry rule opens the other side
        df.loc[df["month_end"] & (df["mom"] < 0), "exit_long"] = 1
        df.loc[df["month_end"] & (df["mom"] > 0), "exit_short"] = 1
        return df

## 10. Part 4: the size of a position
Weight from section 5 × the whole account = the position; margin = position / leverage. S = `current_whitelist()`, the pairs freqtrade actually trades. `TARGET`, `LEV` from section 7, `sigma` from 8.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def leverage(self, pair, **kwargs):
        return self.LEV

    def target_margin(self, pair):
        df, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
        coins = len(self.dp.current_whitelist())       # the paper's S
        weight = self.TARGET / float(df["sigma"].iloc[-1]) / coins
        # weight is a share of the whole account; freqtrade wants margin
        return weight * self.wallets.get_total_stake_amount() / self.LEV

    def custom_stake_amount(self, pair, current_time, current_rate,
                            proposed_stake, min_stake, max_stake, leverage,
                            entry_tag, side, **kwargs):
        return min(self.target_margin(pair), max_stake)

## 11. Part 5: rebalance once a month
On the 1st, bring every open position back to its target: buy the difference (positive) or sell it (negative); skip changes under 10% to save fees. `target_margin` from section 10.

In [ ]:
%%writefile -a user_data/strategies/TSMOM.py

    def adjust_trade_position(self, trade, current_time, current_rate,
                              current_profit, min_stake, max_stake,
                              current_entry_rate, current_exit_rate,
                              current_entry_profit, current_exit_profit,
                              **kwargs):
        # only on the 1st, not on the day it opened, one order at a time
        if current_time.day != 1 or trade.has_open_orders:
            return None
        if trade.open_date_utc.date() == current_time.date():
            return None
        target = self.target_margin(trade.pair)
        now = trade.amount * current_rate / trade.leverage   # margin it holds
        change = target - now
        if abs(change) < 0.1 * target:          # too small to pay a fee for
            return None
        if change > 0:
            return min(change, max_stake)        # buy the difference
        return max(change, -0.999 * now)         # sell it, never all of it

## 12. Run it
From 2 January 2021 — the first day with twelve months of history. Uses `config.json` (6), `TSMOM` (7–11), `verdict`, `equity`, `drawdown` (section 2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s TSMOM --timerange 20210102-20261031 --export trades > run.log 2>&1
both, trades = verdict("TSMOM")
hourly = equity(trades, start="2021-01-02")
both["drawdown, marked to market"] = drawdown(hourly)

## 13. One coin is missing
freqtrade checks every pair against the markets Binance lists **today** — and drops the delisted ones. Our data has the EOS coin; the backtest does not. Uses `run.log` (section 12), `close` (3).

In [ ]:
!grep -o -m1 "Pair EOS.*whitelist" run.log
eos = close["EOS"].dropna()
print("EOS:", eos.index[0].date(), eos.iloc[0], "->", eos.index[-1].date(),
      eos.iloc[-1], f"({eos.iloc[-1] / eos.iloc[0] - 1:.0%})")

## 14. Where the money went
Profit by coin and by side. Uses `trades` (section 12).

In [ ]:
by_side = trades.groupby(["pair", "is_short"])["profit_abs"].sum().unstack()
print(by_side.round(0).rename(columns={False: "long", True: "short"}))
print(trades["exit_reason"].value_counts().to_string())   # how trades ended

## 15. Long or cash
The same rules without shorts: a negative signal means no position. Inherits sections 7–11. Uses `verdict`, `equity`, `drawdown` (2).

In [ ]:
%%writefile user_data/strategies/TSMOMLong.py
from TSMOM import TSMOM

class TSMOMLong(TSMOM):
    SHORTS = False

### Run long or cash
Uses `config.json` (6), `TSMOMLong` (above), `verdict`, `equity`, `drawdown` (2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s TSMOMLong --timerange 20210102-20261031 --export trades > run.log 2>&1
long_only, trades_long = verdict("TSMOMLong")
long_only["drawdown, marked to market"] = drawdown(equity(trades_long,
                                                          start="2021-01-02"))

## 16. Against holding all ten
Buy-and-hold = the ten coins freqtrade traded, equal amounts, rebalanced daily, valued at the daily close. Uses `close` (3), `drawdown`, `equity` (2), `trades`, `trades_long` (12, 15).

In [ ]:
traded = close.drop(columns="EOS").loc["2021-01-02":]   # what freqtrade traded
daily_return = traded.pct_change().mean(axis=1).fillna(0)  # equal amounts
hold = 10_000 * (1 + daily_return).cumprod()
curves = {"buy and hold, 10 coins": hold,
          "TSMOM long and short": equity(trades, start="2021-01-02"),
          "TSMOM long or cash": equity(trades_long, start="2021-01-02")}
for name, curve in curves.items():
    print(f"{name:>23}: final {curve.iloc[-1]:>8.0f}"
          f"  max drawdown {drawdown(curve):.0%}")
plt.figure(figsize=(11, 5))
for name, curve in curves.items():
    plt.plot(curve.index, curve, label=name)
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2021 to 2026")
plt.show()

## 17. Is it the 12 months?
Crypto moves faster than the futures the paper tested — try 1, 3 and 6 months. Each is `TSMOM` with one setting changed (section 7).

In [ ]:
%%writefile user_data/strategies/TSMOMFast.py
from TSMOM import TSMOM

class TSMOM1m(TSMOM):
    LOOKBACK = 30

class TSMOM3m(TSMOM):
    LOOKBACK = 90

class TSMOM6m(TSMOM):
    LOOKBACK = 180

### Run all three
Uses `config.json` (6), `TSMOM1m`, `TSMOM3m`, `TSMOM6m` (above), `verdict`, `equity`, `drawdown` (2).

In [ ]:
fast = {}
for name in ["TSMOM1m", "TSMOM3m", "TSMOM6m"]:
    !freqtrade backtesting --cache none -c config.json -s {name} --timerange 20210102-20261031 --export trades > run.log 2>&1
    fast[name], t = verdict(name)
    hourly = equity(t, start="2021-01-02")
    fast[name]["drawdown, marked to market"] = drawdown(hourly)

## 18. Every run side by side
12m = long and short, 12m long = long or cash; 1m, 3m, 6m = the faster lookbacks; a legend explains every row. Uses `both` (12), `long_only` (15), `fast` (17).

In [ ]:
table = pd.DataFrame({"12m": both, "12m long": long_only, "1m": fast["TSMOM1m"],
                      "3m": fast["TSMOM3m"], "6m": fast["TSMOM6m"]})
table.index = ["start", "final", "CAGR", "drawdown (freqtrade)", "trades",
               "long profit", "short profit", "funding", "fees",
               "drawdown (hourly)"]
pct = ["CAGR", "drawdown (freqtrade)", "drawdown (hourly)"]
shown_table = table.apply(lambda r: r.map(lambda v: f"{v:.0%}" if r.name in pct
                                          else f"{v:,.0f}"), axis=1)
print(shown_table.to_string())
print("money in USDT (Tether, a dollar stablecoin)")
print("CAGR = compound annual growth rate: growth per year")
print("drawdown = deepest fall from a peak; freqtrade: closed trades only,")
print("  hourly: the whole account valued every hour")
print("funding = paid between longs and shorts every 8 hours")
print("fees = the exchange fee, 0.05% of every order")